In [53]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

sp500 = pd.read_html(url, storage_options=headers)[0]
sp500['Symbol'] = sp500['Symbol'].str.replace('.','-')

symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2026-10-10'

# Original analysis window
analysis_start = pd.to_datetime(end_date) - pd.DateOffset(years=8)

# Extended download: +5 years of warm-up for the 60-month rolling
start_date = analysis_start - pd.DateOffset(years=5)

download = yf.download(
    tickers=symbols_list,
    start=start_date,
    end=end_date,
    auto_adjust=False
)
df = download.stack(level=1)
df.index.names = ['date', 'ticker']
df.columns = df.columns.str.lower()

df


[*********************100%***********************]  503 of 503 completed


Price               adj close       close        high         low        open  \
date       ticker                                                               
2013-10-10 A        32.744595   36.473534   36.623749   35.972816   35.972816   
           AAPL     15.058258   17.487143   17.584999   17.394285   17.547144   
           ABBV     27.446535   45.680000   45.680000   44.509998   45.099998   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      26.040781   33.639999   33.669998   33.270000   33.340000   
...                       ...         ...         ...         ...         ...   
2026-10-09 XYZ      77.239998   77.239998   77.680000   75.260002   75.625000   
           YUM     144.820007  144.820007  145.470001  143.039993  144.000000   
           ZBH      89.139999   89.139999   90.209999   88.400002   88.930000   
           ZBRA    387.179993  387.179993  389.010010  381.989990  386.459991   
           ZTS      74.809998   74.809998   74.849998   73.000000   73.040001   

Price                   volume  
date       ticker               
2013-10-10 A         3391967.0  
           AAPL    278602800.0  
           ABBV      3967500.0  
           ABNB            NaN  
           ABT       7731400.0  
...                        ...  
2026-10-09 XYZ       5232500.0  
           YUM       3138600.0  
           ZBH       1604300.0  
           ZBRA       338400.0  
           ZTS       2912100.0  

[1644307 rows x 6 columns]

In [54]:
df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*((np.log(df['adj close'])-np.log(df['open']))**2)

df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])
df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])
df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

def compute_atr(g):
    try:
        atr = pandas_ta.atr(high=g['high'], low=g['low'],
                            close=g['close'], length=14)
        if atr is None:
            raise ValueError
        std = atr.std()
        if std == 0 or pd.isna(std):
            return pd.Series(index=g.index, dtype=float)
        return (atr - atr.mean()) / std
    except Exception:
        return pd.Series(index=g.index, dtype=float)
    
df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

# ---------- MACD (normalized) ----------
def compute_macd(close):
    try:
        m = pandas_ta.macd(close=close, fast=12, slow=26, signal=9)
        if m is None or not isinstance(m, pd.DataFrame):
            raise ValueError("macd returned None")
        macd_line = m.iloc[:, 0]
        std = macd_line.std()
        if std == 0 or pd.isna(std):
            return pd.Series(index=close.index, dtype=float)
        return (macd_line - macd_line.mean()) / std
    except Exception:
        return pd.Series(index=close.index, dtype=float)
        
df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

df['dollar_volume'] = (df['adj close']*df['volume'])/1e6

df

Price               adj close       close        high         low        open  \
date       ticker                                                               
2013-10-10 A        32.744595   36.473534   36.623749   35.972816   35.972816   
           AAPL     15.058258   17.487143   17.584999   17.394285   17.547144   
           ABBV     27.446535   45.680000   45.680000   44.509998   45.099998   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      26.040781   33.639999   33.669998   33.270000   33.340000   
...                       ...         ...         ...         ...         ...   
2026-10-09 XYZ      77.239998   77.239998   77.680000   75.260002   75.625000   
           YUM     144.820007  144.820007  145.470001  143.039993  144.000000   
           ZBH      89.139999   89.139999   90.209999   88.400002   88.930000   
           ZBRA    387.179993  387.179993  389.010010  381.989990  386.459991   
           ZTS      74.809998   74.809998   74.849998   73.000000   73.040001   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2013-10-10 A         3391967.0         -0.003254        NaN       NaN   
           AAPL    278602800.0         -0.008979        NaN       NaN   
           ABBV      3967500.0         -0.094944        NaN       NaN   
           ABNB            NaN               NaN        NaN       NaN   
           ABT       7731400.0         -0.023514        NaN       NaN   
...                        ...               ...        ...       ...   
2026-10-09 XYZ       5232500.0          0.000328  49.552661  4.300954   
           YUM       3138600.0          0.000129  52.416424  4.902887   
           ZBH       1604300.0          0.000203  42.850914  4.462786   
           ZBRA       338400.0          0.000164  65.556435  5.824392   
           ZTS       2912100.0          0.000092  54.404678  4.245182   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2013-10-10 A            NaN       NaN       NaN       NaN     111.068584  
           AAPL         NaN       NaN       NaN       NaN    4195.272858  
           ABBV         NaN       NaN       NaN       NaN     108.894128  
           ABNB         NaN       NaN       NaN       NaN            NaN  
           ABT          NaN       NaN       NaN       NaN     201.331694  
...                     ...       ...       ...       ...            ...  
2026-10-09 XYZ     4.300954  4.300954 -0.489644 -0.289734     404.158289  
           YUM     4.902887  4.902887  1.358960 -1.131692     454.532075  
           ZBH     4.462786  4.462786 -0.095850 -0.964411     143.007301  
           ZBRA    5.824392  5.824392  0.781167  1.226455     131.021710  
           ZTS     4.245182  4.245182 -0.444979 -0.267829     217.854194  

[1644307 rows x 14 columns]

In [55]:
last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open', 'high', 'low', 'close']]


data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
df.unstack()[last_cols].resample('ME').last().stack('ticker')],axis=1)).dropna()

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2013-11-30 A          124.114808   34.401413         -0.004893  63.270905   
           AAPL      4247.126069   17.201073         -0.006529  75.505257   
           ABBV       128.663377   29.110872         -0.099927  53.155667   
           ABT        180.732962   29.562948         -0.026573  67.138978   
           ACGL        14.294877   18.647121         -0.001074  67.950533   
...                          ...         ...               ...        ...   
2026-10-31 XYZ        334.708773   77.239998          0.000328  49.552661   
           YUM        338.325318  144.820007          0.000129  52.416424   
           ZBH        182.891983   89.139999          0.000203  42.850914   
           ZBRA       158.953923  387.179993          0.000164  65.556435   
           ZTS        315.109929   74.809998          0.000092  54.404678   

                     bb_low    bb_mid   bb_high       atr      macd  
date       ticker                                                    
2013-11-30 A       3.487992  3.487992  3.487992 -1.176092  0.059556  
           AAPL    2.809054  2.809054  2.809054 -1.054149 -0.179477  
           ABBV    3.383430  3.383430  3.383430 -1.191439 -0.216912  
           ABT     3.388983  3.388983  3.388983 -1.224424  0.209882  
           ACGL    2.956437  2.956437  2.956437 -1.181688 -0.017745  
...                     ...       ...       ...       ...       ...  
2026-10-31 XYZ     4.300954  4.300954  4.300954 -0.489644 -0.289734  
           YUM     4.902887  4.902887  4.902887  1.358960 -1.131692  
           ZBH     4.462786  4.462786  4.462786 -0.095850 -0.964411  
           ZBRA    5.824392  5.824392  5.824392  0.781167  1.226455  
           ZTS     4.245182  4.245182  4.245182 -0.444979 -0.267829  

[75018 rows x 9 columns]

In [56]:
data['dollar_volume'] = (data.loc[:, 'dollar_volume'].unstack('ticker').rolling(5*12).mean().stack('ticker'))

# Drop the warm-up period, restore your original 8-year window
data = data[data.index.get_level_values('date') >= analysis_start]

data['dollar_volume_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

data = data[data['dollar_volume']<150].drop(['dollar_volume','dollar_volume_rank'],axis=1)

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2018-10-31 A        61.086258         -0.000668  45.962897  4.055975   
           ACGL     26.976992         -0.000335  50.013039  3.249723   
           ADI      72.600151         -0.007979  42.349001  4.225410   
           ADM      37.358688         -0.024716  40.482355  3.625842   
           AEE      51.544498         -0.018491  50.345386  3.943520   
...                       ...               ...        ...       ...   
2026-10-31 UHS     177.350006          0.000191  54.595592  5.155493   
           VMRK     61.160000          0.000216  45.516630  4.074323   
           VTRS     17.639999          0.000433  57.115798  2.865432   
           WRB      71.910004          0.000076  60.048215  4.208861   
           WY       19.100000          0.000375  31.074509  2.919396   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2018-10-31 A       4.055975  4.055975 -0.296802 -0.885965  
           ACGL    3.249723  3.249723 -0.521624 -0.850549  
           ADI     4.225410  4.225410 -0.437213 -0.980079  
           ADM     3.625842  3.625842 -0.435243 -0.728416  
           AEE     3.943520  3.943520  0.001044  0.041440  
...                     ...       ...       ...       ...  
2026-10-31 UHS     5.155493  5.155493  0.744285  0.234649  
           VMRK    4.074323  4.074323 -0.212267 -1.491690  
           VTRS    2.865432  2.865432 -0.620934  0.386914  
           WRB     4.208861  4.208861  1.566432  0.438942  
           WY      2.919396  2.919396 -0.683067 -2.486516  

[19516 rows x 8 columns]

In [57]:
def calculate_returns(df):


    outlier_cutoff = 0.005

    lags = [1, 2, 3, 6, 9, 12]

    for lag in lags:

        df[f'return_{lag}m'] = (df['adj close'].pct_change(lag).pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),upper=x.quantile(1-outlier_cutoff))).add(1).pow(1/lag).sub(1))

    return df

data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2019-10-31 A        72.055336         -0.001006  53.879030  4.257964   
           ACGL     39.709522         -0.000812  54.782555  3.677065   
           ADM      34.378784         -0.011660  59.546507  3.484299   
           AEE      63.644783         -0.015095  52.887798  4.134630   
           AEP      73.833900         -0.023228  56.088937  4.284257   
...                       ...               ...        ...       ...   
2026-10-31 UHS     177.350006          0.000191  54.595592  5.155493   
           VMRK     61.160000          0.000216  45.516630  4.074323   
           VTRS     17.639999          0.000433  57.115798  2.865432   
           WRB      71.910004          0.000076  60.048215  4.208861   
           WY       19.100000          0.000375  31.074509  2.919396   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2019-10-31 A       4.257964  4.257964 -0.646802  0.029114  -0.011484   
           ACGL    3.677065  3.677065 -0.388829 -0.082422  -0.005241   
           ADM     3.484299  3.484299 -0.671449  0.205738   0.023618   
           AEE     4.134630  4.134630 -0.420898 -0.396220  -0.029357   
           AEP     4.284257  4.284257 -0.446137  0.149334   0.007471   
...                     ...       ...       ...       ...        ...   
2026-10-31 UHS     5.155493  5.155493  0.744285  0.234649   0.017732   
           VMRK    4.074323  4.074323 -0.212267 -1.491690   0.018559   
           VTRS    2.865432  2.865432 -0.620934  0.386914  -0.004515   
           WRB     4.208861  4.208861  1.566432  0.438942   0.064702   
           WY      2.919396  2.919396 -0.683067 -2.486516   0.002625   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2019-10-31 A        0.033220   0.030302  -0.005205   0.000271    0.013857  
           ACGL     0.028210   0.025779   0.035977   0.039961    0.032742  
           ADM      0.051124   0.010886  -0.006825  -0.004377   -0.006903  
           AEE      0.006744   0.010908   0.013113   0.014889    0.017728  
           AEP      0.017618   0.026941   0.019158   0.022527    0.024037  
...                      ...        ...        ...        ...         ...  
2026-10-31 UHS      0.020999   0.017729   0.009227  -0.013568   -0.016321  
           VMRK    -0.025284  -0.023431  -0.007399   0.001714    0.006167  
           VTRS     0.033655   0.004014   0.030612   0.036382    0.048389  
           WRB      0.032429  -0.002428   0.014043   0.006587    0.002961  
           WY      -0.091812  -0.083393  -0.037946  -0.029941   -0.012423  

[16074 rows x 14 columns]

In [64]:
factor_data = web.DataReader('F-F_Research_Data_5_Factors_2X3','famafrench',start='2013')[0].drop('RF',axis=1)

factor_data.index = factor_data.index.to_timestamp()

factor_data = factor_data.resample('ME').last().div(100)

factor_data.index.name = 'date'

factor_data = factor_data.join(data['return_1m']).sort_index()

factor_data

Mkt-RF     SMB     HML     RMW     CMA  return_1m
date       ticker                                                   
2019-10-31 A       0.0207  0.0025 -0.0192  0.0043 -0.0103  -0.011484
           ACGL    0.0207  0.0025 -0.0192  0.0043 -0.0103  -0.005241
           ADM     0.0207  0.0025 -0.0192  0.0043 -0.0103   0.023618
           AEE     0.0207  0.0025 -0.0192  0.0043 -0.0103  -0.029357
           AEP     0.0207  0.0025 -0.0192  0.0043 -0.0103   0.007471
...                   ...     ...     ...     ...     ...        ...
2026-09-30 UHS    -0.0107 -0.0460 -0.0155 -0.0322  0.0108   0.024275
           VMRK   -0.0107 -0.0460 -0.0155 -0.0322  0.0108  -0.067240
           VTRS   -0.0107 -0.0460 -0.0155 -0.0322  0.0108   0.073289
           WRB    -0.0107 -0.0460 -0.0155 -0.0322  0.0108   0.001134
           WY     -0.0107 -0.0460 -0.0155 -0.0322  0.0108  -0.177354

[15984 rows x 6 columns]

In [65]:
observations = factor_data.groupby(level=1).size()

valid_stocks = observations[observations >= 10]

factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

factor_data


Mkt-RF     SMB     HML     RMW     CMA  return_1m
date       ticker                                                   
2019-10-31 A       0.0207  0.0025 -0.0192  0.0043 -0.0103  -0.011484
           ACGL    0.0207  0.0025 -0.0192  0.0043 -0.0103  -0.005241
           ADM     0.0207  0.0025 -0.0192  0.0043 -0.0103   0.023618
           AEE     0.0207  0.0025 -0.0192  0.0043 -0.0103  -0.029357
           AEP     0.0207  0.0025 -0.0192  0.0043 -0.0103   0.007471
...                   ...     ...     ...     ...     ...        ...
2026-09-30 UHS    -0.0107 -0.0460 -0.0155 -0.0322  0.0108   0.024275
           VMRK   -0.0107 -0.0460 -0.0155 -0.0322  0.0108  -0.067240
           VTRS   -0.0107 -0.0460 -0.0155 -0.0322  0.0108   0.073289
           WRB    -0.0107 -0.0460 -0.0155 -0.0322  0.0108   0.001134
           WY     -0.0107 -0.0460 -0.0155 -0.0322  0.0108  -0.177354

[15880 rows x 6 columns]

In [67]:
betas = (factor_data.groupby(level=1,group_keys=False).apply(lambda x: RollingOLS(endog=x['return_1m'],exog=sm.add_constant(x.drop('return_1m', axis=1)),window=min(24, x.shape[0]),min_nobs=len(x.columns)+1).fit(params_only=True).params.drop('const',axis=1)))

betas

Mkt-RF       SMB       HML       RMW       CMA
date       ticker                                                  
2019-10-31 A            NaN       NaN       NaN       NaN       NaN
           ACGL         NaN       NaN       NaN       NaN       NaN
           ADM          NaN       NaN       NaN       NaN       NaN
           AEE          NaN       NaN       NaN       NaN       NaN
           AEP          NaN       NaN       NaN       NaN       NaN
...                     ...       ...       ...       ...       ...
2026-09-30 UHS     0.449058  0.445930  0.093325  0.921159 -0.178045
           VMRK    0.944801  0.045588  1.270519 -0.098966  0.006306
           VTRS    0.830418  1.643992 -0.785022  0.436382  1.000140
           WRB     0.401852  0.135906  1.004922  0.048437  0.416654
           WY      1.029270  0.356598  1.832489  0.467352 -0.790408

[15880 rows x 5 columns]

In [70]:
# factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

# data = (data.join(betas.groupby('ticker').shift()))

# data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

data = data.drop('adj close', axis=1)

data = data.dropna()

data.info()

<class 'pandas.DataFrame'>
MultiIndex: 15452 entries, (Timestamp('2019-10-31 00:00:00'), 'ACGL') to (Timestamp('2026-10-31 00:00:00'), 'WY')
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   garman_klass_vol  15452 non-null  float64
 1   rsi               15452 non-null  float64
 2   bb_low            15452 non-null  float64
 3   bb_mid            15452 non-null  float64
 4   bb_high           15452 non-null  float64
 5   atr               15452 non-null  float64
 6   macd              15452 non-null  float64
 7   return_1m         15452 non-null  float64
 8   return_2m         15452 non-null  float64
 9   return_3m         15452 non-null  float64
 10  return_6m         15452 non-null  float64
 11  return_9m         15452 non-null  float64
 12  return_12m        15452 non-null  float64
 13  Mkt-RF            15452 non-null  float64
 14  SMB               15452 non-null  float64
 15  HML               1545